# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [1]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

HypEx 1.0.7 · pandas 2.2.3


### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [2]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [3]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

вариант 30: «Новая главная страница приложения»
6000 клиентов · test 3042 · control 2958


### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [4]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

проверка формата готова


## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [5]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

клиентов на группу: 3000, σ оборота: 3228 ₽, ρ двух прошлых периодов: 0.593
MDE без снижения дисперсии: 233 ₽
MDE с CUPED:                188 ₽
+3 % к среднему обороту:    95 ₽


#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Если показать новую главную страницу приложения подписчикам Базового тарифа, то оборот в партнерских сервисах на клиента вырастет не меньше чем на 188 руб (примерно 6% от среднего) за 4 недели, потому что на новой главной партнерские сервисы заметнее и клиенты чаще к ним переходят. |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Подписка через ее владельца: одна строка это один владелец, а участники подписки (`members` от 0 до 2) видят ту же версию главной. Если делить по отдельным пользователям, участники одной подписки увидят разные версии и будут покупать через общий аккаунт, контроль частично получит воздействие, и эффект может размыться. Анализ ведем на той же единице: метрики считаем на клиента, а не на покупку. |
| группы и доли | Две группы, `test` (новая главная) и `control` (старая), 50/50, по 3000 клиентов. |
| основная метрика | `spend`: оборот в партнерских сервисах на клиента за 4 недели эксперимента, в рублях. Если покупок не было, это 0. |
| метрики-стражи: что не должно ухудшиться | Отток из подписки (`churned`) и обращения в поддержку (`tickets`) на клиента: новый интерфейс может запутать часть пользователей. Ухудшением считаем значимый рост в тесте по сравнению с контролем (p <0,05), отток проверяем χ² по таблице частот, обращения через разность средних с интервалом. Вспомогательная метрика: число покупок (`purchases`). |
| α, мощность, MDE — числами из ячейки выше | α = 0,05, мощность 80%, одно сравнение. MDE 233 рубля без снижения дисперсии и 188 рублей с CUPED (ρ двух прошлых периодов 0,593). Рост на 3 % (это 95 рублей) , вдвое меньше MDE, такой эффект эксперимент скорее всего не заметит. |
| поправка на множественные сравнения | Решение принимаем по одной основной метрике без поправки. Для всех четырех метрик (`spend`, `purchases`, `churned`, `tickets`) поправка Холма. |
| оценщик | Разность средних test минус control после CUPED, ковариата `spend_prev` (оборот за 4 недели до старта), интервал по Уэлчу. Для сверки разность без CUPED и бутстрэп. |
| окно | 4 недели, без досрочной остановки и промежуточных проверок. |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | **Запускаем**, если эффект значим, оценка не меньше 188 рублей, стражи в норме и акция окупается. **Дорабатываем и повторяем**, если эффект значим, но отток или обращения значимо выросли (p < 0,05) или акция не окупается. **Отказываемся**, если эффект незначим и верхняя граница интервала ниже 188 рублей. **Данных недостаточно**, если интервал включает и 0, и 188 рублей: увеличиваем выборку или продлеваем. |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [6]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

{'control': 2958, 'test': 3042}
scipy:    p = 0.2782
pilot.py: p = 0.2782


0.278

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [7]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

spend_prev: test 3105.72 ₽, control 3209.41 ₽
p = 0.2139


0.214

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [8]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

test 3270.32 ₽, control 3301.55 ₽
разность -31.23 ₽, интервал -197.28 … 134.82, p = 0.7124


[-31.23, -197.28, 134.82]

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [9]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value UTest pass  UTest p-value
  spend  test   3301.547255 3270.316686  -31.230569     -0.945937     NOT OK       0.712419     NOT OK       0.191926


[0.7124, 0.1919]

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [10]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print(f"SRM p = {answer_1}, баланс spend_prev p = {answer_2}")
print(f"эффект {effect['diff']:.0f} ₽ [{effect['lo']:.0f}; {effect['hi']:.0f}], MDE {mde_raw:.0f} ₽ / с CUPED {mde_cuped:.0f} ₽")
print(f"t p = {answer_4[0]}, U p = {answer_4[1]}")


SRM p = 0.278, баланс spend_prev p = 0.214
эффект -31 ₽ [-197; 135], MDE 233 ₽ / с CUPED 188 ₽
t p = 0.7124, U p = 0.1919


#### ✍️ Ответ на вопрос 1

Разбиению можно доверять. Размеры групп 3042 и 2958 значимо не отличаются от нашего плана разбиения на 50/50 (SRM, p = 0,278), и до эксперимента группы тратили примерно одинаково: 3106 и 3209 ₽, разница незначима (p = 0,214). Эффект новой главной получился -31 ₽, интервал от -197 до 135 ₽, то есть ноль в него попадает и роста оборота мы не видим. При этом верхняя граница 135 ₽ ниже MDE (233 ₽, с CUPED 188 ₽), поэтому эффект того размера, ради которого запускали тест, интервал исключает. 
t-критерий(p = 0,712) и U-критерий(p = 0,192) дают одинаковый вывод, что значимой разницы нет, а U ближе к порогу, потому что меньше реагирует на клиентов с очень большими тратами


## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [11]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

         control    test    diff      lo      hi       p
metric                                                  
churned   0.0341  0.0375  0.0033 -0.0061  0.0127  0.4875
tickets   0.0510  0.0575  0.0065 -0.0054  0.0183  0.2846
churned     0    1
grp               
control  2857  101
test     2928  114
χ² по оттоку: p = 0.5323


[0.33, 0.532, 0.006, 0.285]

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [12]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

HypEx о поправке: There was less than three groups or multitest method wasn't provided
  feature  control mean  test mean  difference  TTest p-value  holm  rejected
    spend     3301.5473  3270.3167    -31.2306         0.7124   1.0     False
purchases        4.9753     4.9701     -0.0052         0.9616   1.0     False
  churned        0.0341     0.0375      0.0033         0.4875   1.0     False
  tickets        0.0510     0.0575      0.0065         0.2847   1.0     False


[]

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [13]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

theta = 0.605, дисперсия ниже на 35.5%
разность 31.53 ₽, интервал -101.89 … 164.96


[0.605, 35.5, 31.53, -101.89, 164.96]

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [14]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)
ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

target covariate variance_reduction
 spend spend_lag              35.5%
    feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value
      spend  test   3301.547255 3270.316686  -31.230569     -0.945937     NOT OK       0.712419
spend_cuped  test   3269.725544 3301.259692   31.534148      0.964428     NOT OK       0.643230


31.53

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [15]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

w3 = effect["hi"] - effect["lo"]
w7 = effect_cuped["hi"] - effect_cuped["lo"]
gap_prev = test["spend_prev"].mean() - control["spend_prev"].mean()

print(f"отток: control {guards.loc['churned', 'control']:.2%}, test {guards.loc['churned', 'test']:.2%}, "
      f"разница {guards.loc['churned', 'diff'] * 100:.2f} п.п., p χ² = {p_churn:.3f}")
print(f"обращения: control {guards.loc['tickets', 'control']:.4f}, test {guards.loc['tickets', 'test']:.4f}, "
      f"разница {guards.loc['tickets', 'diff']:.4f}, p = {guards.loc['tickets', 'p']:.3f}")
print("значимо после Холма:", answer_6)
print(f"ширина интервала: без CUPED {w3:.0f} ₽, с CUPED {w7:.0f} ₽, уже на {1 - w7 / w3:.1%}")
print(f"оценка: без CUPED {effect['diff']:.2f} ₽, с CUPED {effect_cuped['diff']:.2f} ₽")
print(f"spend_prev: test {test['spend_prev'].mean():.0f} ₽, control {control['spend_prev'].mean():.0f} ₽, "
      f"разница {gap_prev:.0f} ₽, theta * разница = {theta * gap_prev:.0f} ₽")

отток: control 3.41%, test 3.75%, разница 0.33 п.п., p χ² = 0.532
обращения: control 0.0510, test 0.0575, разница 0.0065, p = 0.285
значимо после Холма: []
ширина интервала: без CUPED 332 ₽, с CUPED 267 ₽, уже на 19.6%
оценка: без CUPED -31.23 ₽, с CUPED 31.53 ₽
spend_prev: test 3106 ₽, control 3209 ₽, разница -104 ₽, theta * разница = -63 ₽


#### ✍️ Ответ на вопрос 2

По результатам выше можно сказать, что стражи не сработали: отток в тесте выше на 0,33 пп. (3,75% против 3,41%, p χ² = 0,532), обращений больше на 0,0065 на клиента (при p =0,285), а после поправки Холма не осталось ни одной значимой метрики, все p стали равны 1. 
При этом CUPED снизил дисперсию на 35,5% и сузил интервал на 19,6% с 332 до 267 рублей, а сама оценка поменяла знак (с -31 до +32 рублей), и это может объясняться прошлым периодом (еще до эксперимента тест тратил на 104 руб меньше контроля (3106 против 3209 рублей), и CUPED вычел эту случайную разницу (0,605*104 ≈63 рубля). 
Вывод от этого не изменился - ноль по-прежнему внутри интервала от -102 до 165 рублей, а верхняя граница находится ниже MDE с CUPED 188 рублей

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [16]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.
rng = np.random.default_rng(VARIANT)
x_test = test["spend"].to_numpy()
x_control = control["spend"].to_numpy()

boot = []
for _ in range(2000):
    s_test = rng.choice(x_test, size=len(x_test))
    s_control = rng.choice(x_control, size=len(x_control))
    boot.append(s_test.mean() - s_control.mean())

boot_lo, boot_hi = np.percentile(boot, [2.5, 97.5])

print(f"бутстреп:  {boot_lo:.2f} … {boot_hi:.2f}, ширина {boot_hi - boot_lo:.2f} ₽")
print(f"задача 3:  {effect['lo']:.2f} … {effect['hi']:.2f}, ширина {effect['hi'] - effect['lo']:.2f} ₽")

answer_9 = [round(float(boot_lo), 2), round(float(boot_hi), 2)]
answer_9


бутстреп:  -193.06 … 132.84, ширина 325.90 ₽
задача 3:  -197.28 … 134.82, ширина 332.10 ₽


[-193.06, 132.84]

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [17]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.
candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}

rho2 = pd.Series({name: mine["spend"].corr(x) ** 2 * 100 for name, x in candidates.items()})
rho2_purchases = mine["spend"].corr(mine["purchases"]) ** 2 * 100

print(rho2.sort_values(ascending=False).round(1).to_string())
print(f"purchases (во время эксперимента, брать нельзя): {rho2_purchases:.1f} %")

best = rho2.idxmax()
answer_10 = [best, round(float(rho2[best]), 1)]
answer_10

spend_prev+spend_prev2    40.1
sessions_prev             38.9
sessions_prev2            35.5
spend_prev                35.5
purchases_prev            29.0
spend_prev2               28.6
purchases (во время эксперимента, брать нельзя): 72.0 %


['spend_prev+spend_prev2', 40.1]

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [19]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.
feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]

rct = mine.assign(t=lambda d: (d["grp"] == "test").astype(int))
res_match = Matching(n_neighbors=5).execute(Dataset(
    roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
           **{f: FeatureRole() for f in feats}},
    data=rct, default_role=InfoRole()))
match_resume = F(res_match.resume)
print(match_resume.to_string())
print(match_resume.index.tolist(), match_resume.columns.tolist())

ate = match_resume.loc["ATE", ["Effect Size", "CI Lower", "CI Upper"]]
answer_11 = [round(float(x), 2) for x in ate]
answer_11

     Effect Size  Standard Error  P-value  CI Lower  CI Upper outcome
ATT        61.97           73.97     0.40    -83.01    206.94   spend
ATC        55.57           73.52     0.45    -88.52    199.67   spend
ATE        58.82           71.17     0.41    -80.68    198.31   spend
['ATT', 'ATC', 'ATE'] ['Effect Size', 'Standard Error', 'P-value', 'CI Lower', 'CI Upper', 'outcome']


[58.82, -80.68, 198.31]

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [20]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнерских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, руб
CHURN_COST = 750      # потеря от ушедшего подписчика (три месяца «Базового», в руб)
lift = effect_cuped["diff"]                       # эффект с CUPED, задача 7
ticket_gap = guards.loc["tickets", "diff"]        # разница обращений, задача 5
churn_gap = guards.loc["churned", "diff"]         # разница оттока, задача 5

money = pd.Series({
    "комиссия с прироста оборота": lift * COMMISSION,
    "дополнительные обращения": -ticket_gap * TICKET_COST,
    "дополнительный отток": -churn_gap * CHURN_COST,
})
money["итого на клиента за 4 недели"] = money.sum()
print(money.round(2).to_string())

answer_12 = round(float(money["итого на клиента за 4 недели"]), 2)
answer_12

комиссия с прироста оборота     0.47
дополнительные обращения       -0.97
дополнительный отток           -2.50
итого на клиента за 4 недели   -3.00


-3.0

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [21]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

widths = pd.DataFrame({
    "нижняя": [effect["lo"], effect_cuped["lo"], boot_lo, ate.iloc[1]],
    "верхняя": [effect["hi"], effect_cuped["hi"], boot_hi, ate.iloc[2]],
}, index=["3. разность средних", "7. CUPED", "9. бутстреп", "11. мэтчинг"])
widths["ширина"] = widths["верхняя"] - widths["нижняя"]
print(widths.round(0).to_string())

print(f"ρ²: лучший кандидат {best} {rho2[best]:.1f} %, spend_prev {rho2['spend_prev']:.1f} %, purchases {rho2_purchases:.1f} %")

losses = money["дополнительные обращения"] + money["дополнительный отток"]
print(f"итог: {money.iloc[-1]:.2f} ₽; на границах интервала CUPED: "
      f"{effect_cuped['lo'] * COMMISSION + losses:.2f} … {effect_cuped['hi'] * COMMISSION + losses:.2f} ₽")
print(f"для окупаемости нужен прирост оборота {-losses / COMMISSION:.0f} ₽ на клиента, получили {lift:.0f} ₽")

                     нижняя  верхняя  ширина
3. разность средних  -197.0    135.0   332.0
7. CUPED             -102.0    165.0   267.0
9. бутстреп          -193.0    133.0   326.0
11. мэтчинг           -81.0    198.0   279.0
ρ²: лучший кандидат spend_prev+spend_prev2 40.1 %, spend_prev 35.5 %, purchases 72.0 %
итог: -3.00 ₽; на границах интервала CUPED: -5.00 … -1.00 ₽
для окупаемости нужен прирост оборота 231 ₽ на клиента, получили 32 ₽


#### ✍️ Ответ на вопрос 3
Я больше всего доверяю оценке с CUPED (+32 рубля, интервал от -102 до 165 рублей), так как у нее самый узкий интервал (267 руб против 332 руб у обычной разности, 326 руб у бутстрепа и 279 руб у метчинга), потому что CUPED убирает часть разброса, которую объясняют траты клиента до эксперимента, и также поправляет случайный перекос групп до старта. 
Бутстреп почти совпал с обычным интервалом, а значит при 3000 клиентов в группе формула интервала из diff_ci дает правильный результат, несмотря на тяжелый хвост, а метчинг при случайном делении ничего не исправляет и только добавляет шум. 

Покупки во время эксперимента дают самый большой ρ² (72% против 40,1% у суммы двух прошлых периодов), но на них уже повлияла новая главная, и CUPED вместе с дисперсией вычел бы сам эффект. При наших допущениях акция не окупается: -3 руб на клиента за 4 недели, даже на верхней границе интервала около -1 рубля, а для окупаемости нужен прирост оборота 231 руб против полученных 32 руб; 

более того, в расчете при этом нет стоимости разработки страницы, эффекта после 4 недель и того, что рост оттока и обращений незначим и может быть случайным

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** мы хотели узнать, поднимет ли новая главная страница оборот подписчиков в партнерских сервисах. Заранее договорились считать ее успешной, если за 4 недели оборот вырастет хотя бы на 188 руб на клиента (около 6%) и при этом не вырастет отток и обращения в поддержку. В эксперименте участвовали 6000 подписчиков, из которых одна половина видела новую главную, вторая половина - старую. Группы разделились практически поровну и их размеры (3042 и 2958) значимо не отличаются от плана 50/50 (p= 0,278), а до эксперимента клиенты в обеих группах тратили одинаково (p= 0,214).

**Что получилось:** роста оборота мы не увидели. С учетом того, сколько клиенты тратили до эксперимента, разница составила +32 рубля на клиента за 4 недели, 95 % доверительный интервал от -102 до +165 руб (то есть он включает ноль). Важно, что даже верхняя граница в 165 руб ниже нашего порога в 188 рублей, поэтому эффекта того размера, на который мы рассчитывали, у новой главной страницы нет. Отток вырос с 3,41 до 3,75%, обращений стало немного больше (0,058 против 0,051 на клиента); при этом оба изменения могут быть случайными, но оба изменились в худшую сторону. В деньгах это -3 рубля на клиента за 4 недели (так как комиссия с прироста оборота дает +0,47 руб, а лишние обращения и отток стоят -0,97 и -2,50 руб). Чтобы выйти в ноль, оборот должен был вырасти примерно на 231 рубль.

**Какие остались вопросы:** небольшой рост, напримерр на 3% (95 руб), эксперимент исключить не может, так как чтобы его заметить, нужно взять в тест больше клиентов. Также неясно случаен ли рост оттока и обращений - данные допускают, что отток на самом деле вырос до 1,3 п.п. Мы наблюдали только первые 4 недели и не знаем, как изменится поведение, когда пользователи привыкнут к новому интерфейсу. Кроме того, денежный расчет опирается на допущения (комиссия в 1,5%, обращение 150 руб, ушедший клиент 750 руб) и не учитывает стоимость разработки страницы.

**Что рекомендуем:** не запускать новую главную на всех ради роста оборота. По правилу, которое мы зафиксировали до эксперимента, это случай отказа в запуске, потому что эффект незначим, и даже его верхняя граница ниже порога 188 руб. Финансовый результат тоже отрицательный, причем даже в самом оптимистичном сценарии мы получили около -1 рубля на клиента. Если новая главная страница нужна по другим причинам (например ради удобства интерфейса) ее стоит доработать и проверить повторно, отдельно следя за оттоком и обращениями

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [22]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

задача  1: 0.278
    ✅ формат в порядке
задача  2: 0.214
    ✅ формат в порядке
задача  3: [-31.23, -197.28, 134.82]
    ✅ формат в порядке
задача  4: [0.7124, 0.1919]
    ✅ формат в порядке
задача  5: [0.33, 0.532, 0.006, 0.285]
    ✅ формат в порядке
задача  6: []
    ✅ формат в порядке
задача  7: [0.605, 35.5, 31.53, -101.89, 164.96]
    ✅ формат в порядке
задача  8: 31.53
    ✅ формат в порядке
задача  9: [-193.06, 132.84]
    ✅ формат в порядке
задача 10: ['spend_prev+spend_prev2', 40.1]
    ✅ формат в порядке
задача 11: [58.82, -80.68, 198.31]
    ✅ формат в порядке
задача 12: -3.0
    ✅ формат в порядке

Не забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.

--- служебная строка, не удаляйте ---
HW3_ANSWERS={"variant": 30, "answers": {"1": 0.278, "2": 0.214, "3": [-31.23, -197.28, 134.82], "4": [0.7124, 0.1919], "5": [0.33, 0.532, 0.006, 0.285], "6": [], "7": [0.605, 35.5, 31.53, -101.89, 164.96], "8": 31.53, "9": [-193.06, 132.84], "10": ["spend_prev+spend

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [24]:
# Исследование А: ваш код. Всё — от mine.

#Правильный ответ мы знаем из рандомизации, поэтому можем проверить насколько врет наивное сравнение и помогает ли мэтчинг. 
feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]  # признаки задачи 11

# Шаг 1. Наблюдательные данные 
# Активные: сессий до эксперимента не меньше медианы по варианту
# Из теста берем только активных (они увидели акцию), из контроля всех
med = mine["sessions_prev"].median()
active_test = test[test["sessions_prev"] >= med]
active_control = control[control["sessions_prev"] >= med]

obs = pd.concat([active_test, control])
obs = obs.assign(t=(obs["grp"] == "test").astype(int))   # группа числом: 1 тест, 0 контроль
obs = obs.sort_values("client_id", ignore_index=True)    # один и тот же порядок строк при каждом запуске

#  Шаг 2. считаем разность 
# так посчитал бы продакт (активные с акцией против всех без нее)
# жду завышения - активные клиенты и без акции тратят больше
naive = diff_ci(active_test["spend"], control["spend"])


#  Шаг 3. Мэтчинг, ATT 
# Каждому активному из теста подбираем 5 самых похожих клиентов из контроля по признакам до эксперимента
#ATT -эффект на тех, кто акцию увидел 
def att_matching(features):
    res = Matching(n_neighbors=5).execute(Dataset(
        roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
               **{f: FeatureRole() for f in features}},
        data=obs, default_role=InfoRole()))
    row = F(res.resume).loc["ATT", ["Effect Size", "CI Lower", "CI Upper"]]
    return [float(x) for x in row], res


att_full, res_full = att_matching(feats)

# Шаг 4. Эталон 
# Активные из теста / активных из контроля. Внутри активных группы случайные, поэтому это честная оценка эффекта на активных
truth = diff_ci(active_test["spend"], active_control["spend"])

# Шаг 5. Мэтчинг без sessions_prev
# Убираем признак, по которому шел отбор
att_short, _ = att_matching([f for f in feats if f != "sessions_prev"])

# выводим сводную таблицу 
research_a = pd.DataFrame(
    [[naive["diff"], naive["lo"], naive["hi"]],
     att_full,
     att_short,
     [truth["diff"], truth["lo"], truth["hi"]]],
    index=["2. наивная разность",
           "3. мэтчинг ATT, все признаки",
           "5. мэтчинг ATT, без sessions_prev",
           "4. эталон: активные test - активные control"],
    columns=["оценка", "нижняя", "верхняя"])
research_a["смещение от эталона"] = research_a["оценка"] - truth["diff"]

print(f"медиана sessions_prev = {med:.0f}; активных в тесте {len(active_test)}, в контроле {len(active_control)}")
print(research_a.round(1).to_string())

# Направление 1 - близнецы для теста (ATT). p > 0,05- различий по признаку тест не нашел
quality = F(res_full.quality_results)
quality["direction"] = quality["group"].str[0]
print(quality.pivot_table(index="feature", columns="direction",
                          values=["TTest p-value", "KSTest p-value"]).round(4))

медиана sessions_prev = 15; активных в тесте 1509, в контроле 1539
                                             оценка  нижняя  верхняя  смещение от эталона
2. наивная разность                          1438.4  1211.1   1665.7               1289.4
3. мэтчинг ATT, все признаки                  242.1    23.4    460.7                 93.0
5. мэтчинг ATT, без sessions_prev             741.0   519.8    962.2                592.0
4. эталон: активные test - активные control   149.0  -122.4    420.5                  0.0
              KSTest p-value         TTest p-value        
direction                  0       1             0       1
feature                                                   
age                   0.2174  0.9692        0.3384  0.7243
kyc_level             1.0000  1.0000        0.9851  0.8550
members               1.0000  1.0000        0.9340  0.9441
sessions_prev         0.0000  0.0000        0.0000  0.0013
spend_prev            0.0000  0.0288        0.0000  0.0236
tenure_days

#### ✍️ Исследование А

**Вопрос.** Если бы рандомизации не было и новую главную страницу увидели только активные клиенты насколько наивное сравнение с остальными завысило бы эффект и помог бы это исправить метчинг?

**Как проверяли.** Сначала я выделила активных клиентов (тех, у кого до эксперимента было не меньше медианы по количеству сессий, у меня это 15). Дальше собрала "наблюдательные" данные, как будто акцию увидели только они: из теста взяла 1509 активных, из контроля всех. На этих данных посчитала наивную разность средних оборота и ATT мэтчингом с 5 соседями по признакам задачи 11, а потом повторила мэтчинг без sessions_prev. Чтобы понять, кто ближе к правильному результату, нужен был эталон: для этого я сравнила активных из теста с активными из контроля (1539 человек). Внутри активных группы случайные, так что эту оценку дает сама рандомизация.

**Что получилось.** Наивное сравнение дало 1438 руб, тогда как эталон всего 149 руб (интервал от -122 до 420 руб), т.е эффект завышен почти в 10 раз. Почти вся эта разница объясняется не акцией, а тем, что активные клиенты и так тратят больше. Мэтчинг по всем признакам снизил оценку до 242 руб и убрал около 93% смещения, однако его интервал от 23 до 461 руб не включает ноль, и, опираясь на него, мы бы сделали неверный вывод о значимом эффекте. Когда я убрала из признаков sessions_prev, оценка выросла до 741 руб, а интервал от 520 до 962 руб перестал включать эталонную оценку в 149 руб; до наивной оценки она не дошла, потому что часть информации об активности осталась в spend_prev, который связан с числом сессий. Проверка качества подбора также  объясняет и остаточное смещение в 93 руб (даже при полном наборе признаков пары заметно расходятся как раз по sessions_prev и spend_prev (p < 0,03), то есть подобрать по-настоящему похожих "близнецов" по признаку отбора не удалось).

**Что это значит и чего расчет не доказывает.** Мэтчинг справляется с отбором гораздо лучше наивного сравнения, но работает только при трех условиях: все причины отбора есть среди признаков (условная независимость), для каждого клиента из теста в контроле находятся похожие (пересечение), и само смещение не слишком велико. В моем случае условия были почти идеальными: я точно знала, что отбор шел по сессиям и этот признак был в данных. Тем не менее мэтчинг не устранил смещение полностью и привел бы к неверному выводу о значимости. Расчет не доказывает, что мэтчинг сработает так же в реальной задаче, тк там отбор может идти по тому, чего в данных нет, например по интересу к новинкам, и тогда интервал будет узким, но далеким от настоящего эффекта (причем по самим данным это не проверить). Кроме того, это один вариант и одно правило отбора, а эталон тоже оценен с погрешностью, поэтому эффект надежнее оценивать в рандомизированном эксперименте, а мэтчинг использовать как приближение там, где рандомизация невозможна

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [25]:
# Исследование Б: ваш код. Всё — от mine.

# Вопрос: сколько клиентов нужно, чтобы заметить рост оборота на 3%?
# Формула мощности предполагает нормальность, а у оборота тяжелый хвост, поэтому проверяем формулу симуляцией
from statistics import NormalDist

rng = np.random.default_rng(VARIANT)        # seed = номер варианта: повторный запуск дает те же числа

#  Шаг 1. Оборот контрольной группы 
# В контроле эффекта нет, поэтому эффект мы внесем сами и будем точно знать его размер
y_control = control["spend"].reset_index(drop=True)
x_control = control["spend_prev"].reset_index(drop=True)

effect_3pct = 0.03 * y_control.mean()        # +3% к среднему обороту контроля, в руб
sizes_n = [1_000, 3_000, 10_000, 20_000, 40_000]
K = 500                                      # сколько раз проводим эксперимент для каждого n


# Шаг 2. Мощность симуляцией
# Каждый повтор берем 2n клиентов с возвращением, к первой половине прибавляем эффект,
# сравниваем половины t-критерием Уэлча. Доля повторов с p < 0,05 и есть мощность
def simulated_power(y, n):
    hits = 0
    for _ in range(K):
        sample = rng.choice(y, size=2 * n)
        group_a = sample[:n] + effect_3pct   # "тест" с внесенным эффектом
        group_b = sample[n:]                 # "контроль"
        hits += stats.ttest_ind(group_a, group_b, equal_var=False).pvalue < 0.05
    return hits / K


# Шаг 3. То же после CUPED 
# Скорректированный оборот считаем сразу для каждого клиента контроля (theta тоже по контролю) и уже его выбираем с возвращением. 
# Если выбирать spend и spend_prev по отдельности, связь между ними пропадет и CUPED ничего не даст
y_control_cuped, theta_control = cuped(y_control, x_control)

# Шаг 4. Мощность 
# Phi(эффект / (sigma * sqrt(2/n)) - z_{1-alpha/2})
z_alpha = NormalDist().inv_cdf(0.975)


def formula_power(sigma, n):
    return NormalDist().cdf(effect_3pct / (sigma * (2 / n) ** 0.5) - z_alpha)
    
# Считаем симуляцию и формулу для обоих вариантов метрики и всех n
rows = []
for label, y in [("без CUPED", y_control.to_numpy()), ("с CUPED", y_control_cuped.to_numpy())]:
    sigma = y.std(ddof=1)
    for n in sizes_n:
        rows.append({"метрика": label, "n на группу": n,
                     "симуляция": simulated_power(y, n),
                     "формула": formula_power(sigma, n)})
research_b = pd.DataFrame(rows)

#  Шаг 5. Сколько клиентов нужно и хватило бы нашего эксперимента 
# Число клиентов для 80 % мощности по формуле наша n_per_group из pilot.py
n_need_raw = n_per_group(y_control.std(), effect_3pct)
n_need_cuped = n_per_group(y_control_cuped.std(), effect_3pct)

print(f"эффект +3 % = {effect_3pct:.1f} руб, theta по контролю = {theta_control:.3f}")
print(f"по формуле для 80 % нужно на группу: без CUPED {n_need_raw}, с CUPED {n_need_cuped}")
print(f"в нашем эксперименте на группу: test {len(test)}, control {len(control)}")
print(f"мощность по формуле при n = {len(control)}: без CUPED {formula_power(y_control.std(), len(control)):.1%}, "
      f"с CUPED {formula_power(y_control_cuped.std(), len(control)):.1%}")
print(research_b.pivot(index="n на группу", columns="метрика",
                       values=["симуляция", "формула"]).round(3))


эффект +3 % = 99.0 руб, theta по контролю = 0.577
по формуле для 80 % нужно на группу: без CUPED 17080, с CUPED 11158
в нашем эксперименте на группу: test 3042, control 2958
мощность по формуле при n = 2958: без CUPED 21.4%, с CUPED 30.2%
            симуляция           формула        
метрика     без CUPED с CUPED без CUPED с CUPED
n на группу                                    
1000            0.100   0.158     0.100   0.131
3000            0.206   0.302     0.216   0.306
10000           0.580   0.766     0.573   0.756
20000           0.864   0.956     0.858   0.963
40000           0.994   1.000     0.990   1.000


#### ✍️ Исследование Б

**Вопрос.** Сколько клиентов нужно на группу, чтобы с мощностью 80% заметить рост оборота на 3%, и можно ли доверять формуле мощности, если у оборота тяжелый хвост?

**Как проверяли.** Я взяла оборот контрольной группы, где эффекта точно нет, и создала эффект сама (задала 3% от среднего, то есть 99 руб). Для каждого размера группы от 1000 до 40000 клиентов я 500 раз набирала с возвращением 2n значений, к первой половине прибавляла 99 руб и сравнивала половины t-критерием Уэлча (доля случаев, когда критерий нашел различие, и есть мощность). Затем повторила то же для оборота после CUPED: сначала посчитала скорректированный оборот каждого клиента контроля (θ= 0,577), а потом выбирала уже его, чтобы не потерять связь с прошлым периодом. Результаты сравнила с формулой мощности из семинара 7.

**Что получилось.** По формуле для 80% нужно 17080 клиентов на группу без CUPED и 11158 с CUPED, то есть CUPED сокращает нужную выборку примерно на треть. Симуляция говорит то же самое: без CUPED мощность 58% при 10000 и 86% при 20000, с CUPED уже 77% при 10000. В нашем эксперименте было около 3000 клиентов на группу, и при таком размере мощность для роста на 3% всего 21% без CUPED и 30% с CUPED. Симуляция и формула почти везде совпали с точностью до 1 п.п.; самое заметное расхождение при 1000 клиентах с CUPED (15,8% против 13,1%), но при 500 повторах погрешность самой симуляции около 2 п.п., так что это в пределах шума.

**Что это значит и чего расчет не доказывает.** Нашего эксперимента не хватило бы, чтобы заметить рост на 3%: даже если такой эффект есть, мы увидели бы его примерно в одном случае из трех, а для надежного результата нужно в 4-6 раз больше клиентов. Тяжелый хвост распределения оборота на точность формулы здесь почти не повлиял, уже при тысяче клиентов на группу среднее распределено почти нормально. При этом расчет не доказывает, что формуле можно доверять всегда (!!). Я выбирала клиентов только из своей выборки 2958 чел, и если в реальной аудитории встречаются клиенты с гораздо большими тратами, симуляция их не учтет. Кроме того, я добавляла всем одинаковые 99 руб, а в жизни эффект может касаться только части клиентов или зависеть от их трат, и тогда реальная мощность окажется ниже.